# Understanding BERT — a beginner-friendly walkthrough

**Paper:** *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*
Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova (Google AI Language, 2018) — [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)

---

### How to use this notebook

The notebook has two halves:

| Part | What you get | Code? |
|---|---|---|
| **A. Understanding the paper** (Sections 1–10) | The ideas, explained simply, in the same order the paper builds them | No code — just reading |
| **B. Using BERT in practice** (Sections 11–17) | Short, runnable demos where you *see* each idea from the paper working | Small snippets using the Hugging Face library |

We **don't** build BERT from scratch. The paper itself says its architecture is "almost identical" to the standard Transformer, so re-implementing it wouldn't teach you what's actually new. What's new is *how BERT is trained* and *how it's reused* — that's what we focus on.

At the end of Part A there are **check-your-understanding questions**. Try answering before clicking to reveal the answer — that's where the learning actually sticks.

### The one-sentence summary

> BERT reads a huge amount of plain text and learns language by playing a **fill-in-the-blank game** (looking at words on *both* sides of the blank), and then that one pre-trained model can be quickly adapted to almost any language task by adding **one small layer** on top.

# PART A — Understanding the paper

## 1. The problem BERT is trying to solve

### Labeled data is expensive, plain text is free

Most language tasks — sentiment analysis, question answering, spotting names in text — need **labeled** examples (a human has to mark the right answer). Labeled datasets are small and costly. But **unlabeled** text (Wikipedia, books) is almost unlimited.

So the idea, which existed before BERT, is **pre-training**:

1. **Pre-train**: teach a model general knowledge of language using huge amounts of unlabeled text.
2. **Transfer**: reuse that knowledge on a specific task that only has a little labeled data.

**Analogy:** You don't teach someone to be a lawyer starting from the alphabet. They first spend years learning to read and understand language in general (pre-training), and then a few years of law school (task-specific training). BERT does the same.

### Two existing ways to reuse a pre-trained model

The paper describes two strategies that existed in 2018:

| Strategy | How it works | Example |
|---|---|---|
| **Feature-based** | Run text through the pre-trained model, take its output vectors as *extra features*, and feed them into a separate model you design for your task. The pre-trained model is frozen. | **ELMo** |
| **Fine-tuning** | Take the pre-trained model, add a tiny output layer, and continue training *all* of it on your task. | **OpenAI GPT** |

BERT belongs to the **fine-tuning** family.

### What was wrong with the existing models?

They were **unidirectional** — they read text in only one direction.

- **GPT** reads left-to-right. When it builds its understanding of a word, it can only look at the words *before* it.
- **ELMo** trains one left-to-right model and a separate right-to-left model, then glues their outputs together. Each half only ever sees one side; they never think *jointly*.

Why does that matter? Look at this:

> "I went to the **bank** to deposit my paycheck."
> "I went to the **bank** to go fishing."

Reading left-to-right, when you reach "bank" both sentences are *identical* ("I went to the bank"). The clue that tells you which "bank" it is comes **after** the word. A left-to-right model has to decide what "bank" means without that clue.

**BERT's core claim:** a model that looks at the left and right context **at the same time, in every layer** ("deeply bidirectional") understands language better. The rest of the paper is about (1) how to make that possible and (2) proving it works.

## 2. Background you need (the short version)

### 2.1 Tokens and embeddings

A model can't read letters. Text is split into **tokens** (words or pieces of words), and each token is turned into a list of numbers called an **embedding** (a vector). In BERT-Base each token becomes a vector of **768 numbers**.

A good embedding puts similar meanings close together: "king" and "queen" end up near each other, "king" and "banana" far apart.

### 2.2 Contextual embeddings

Old-style embeddings (like word2vec) give each word **one fixed vector**. "bank" gets the same vector in both sentences above — clearly not ideal.

A **contextual** model gives each word a vector that depends on the sentence around it. BERT produces contextual embeddings: "bank" in the money sentence gets a different vector than "bank" in the river sentence. (You'll see this happen in Section 14.)

### 2.3 The Transformer encoder and self-attention

BERT is built from a stack of **Transformer encoder layers** (from the 2017 paper *Attention Is All You Need*). You don't need the full details, just the intuition:

**Self-attention** = every word looks at every other word in the sentence and decides how much each one matters to it, then mixes in information from the important ones.

In "The animal didn't cross the street because **it** was too tired", self-attention lets the word "it" pay strong attention to "animal", so the vector for "it" absorbs the meaning "this refers to the animal".

Stack 12 of these layers and each word's vector gets progressively richer: early layers capture simple things (grammar, nearby words), later layers capture more abstract meaning.

<details>
<summary><b>For the curious: the actual attention formula (optional)</b></summary>

Each token's vector is turned into three vectors: a **Query** (what am I looking for?), a **Key** (what do I contain?), and a **Value** (what information do I pass on?). Attention is

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

The $QK^\top$ part scores how well each word's query matches every other word's key; softmax turns the scores into weights that sum to 1; the weights are used to average the values. "Multi-head" attention just runs several of these in parallel so the model can track different kinds of relationships at once.
</details>

### 2.4 Why "encoder" vs "decoder" matters here

The paper's footnote makes a useful point:

- A Transformer where every word can attend to **all** words (left and right) is called an **encoder**. → BERT
- A Transformer where every word can attend only to words on its **left** is called a **decoder**, because that restriction is what lets it *generate* text one word at a time. → GPT

So BERT is great at **understanding** text, but it is not designed to **write** text the way GPT does.

## 3. BERT's big idea — and the obstacle it had to get around

### The obstacle: "seeing itself"

The classic way to pre-train a language model is: *predict the next word*. Given "the cat sat on the", predict "mat". This works naturally left-to-right.

Why not just make it bidirectional? Because if every word can see every other word, the task becomes trivial cheating. If the model is supposed to predict word #5, and word #5 is sitting right there in the input, the model can just copy it. The paper describes this as each word being able to indirectly "see itself".

### The fix: the fill-in-the-blank game (Masked Language Model)

Instead of predicting the *next* word, **hide some words and predict the hidden ones**:

> Input: `the cat [MASK] on the mat`
> Task: predict that `[MASK]` = `sat`

Now the model can freely look left *and* right — the answer isn't in the input, so there's nothing to copy. This is called a **Masked Language Model (MLM)**. The idea comes from a 1953 reading-comprehension test called the **Cloze task**, where people fill in deleted words.

That's the heart of the paper. Everything else is design detail.

### The full recipe in two steps

```
                STEP 1: PRE-TRAINING                          STEP 2: FINE-TUNING
          (once, huge cost, unlabeled text)            (per task, cheap, labeled data)

   Wikipedia + Books (3.3 billion words)            Your task data (e.g. 5,000 reviews)
                  │                                               │
                  ▼                                               ▼
     ┌──────────────────────────┐                   ┌──────────────────────────┐
     │  BERT learns language    │   copy weights    │  Same BERT + 1 tiny new  │
     │  via two games:          │ ───────────────►  │  output layer, all of it │
     │   • fill in the blank    │                   │  trained a little more   │
     │   • is B the next        │                   │                          │
     │     sentence after A?    │                   │                          │
     └──────────────────────────┘                   └──────────────────────────┘
```

A distinctive point the paper stresses: the architecture is **the same** in both steps. Only the very last output layer changes.

## 4. The architecture (just the numbers)

BERT is a stack of Transformer encoder layers. The paper uses three letters:

- **L** = number of layers (Transformer blocks stacked on top of each other)
- **H** = hidden size (how many numbers represent each token)
- **A** = number of attention heads (how many parallel "attention patterns" per layer)

| Model | L (layers) | H (hidden size) | A (heads) | Parameters |
|---|---|---|---|---|
| **BERT-Base** | 12 | 768 | 12 | 110 million |
| **BERT-Large** | 24 | 1024 | 16 | 340 million |

Two details worth knowing:

- **BERT-Base was made deliberately the same size as OpenAI GPT**, so the comparison is fair. The *only* major architectural difference between them is that GPT's attention is restricted to the left; BERT's isn't. So when BERT wins, it's because of bidirectionality and the training tasks — not because it's bigger.
- Inside each layer there's a small feed-forward network of size 4H (3072 for Base). The activation function is **GELU** instead of ReLU (following GPT).

## 5. How BERT reads its input

BERT needs to handle very different tasks — some take one piece of text (a review to classify), some take two (a question and a passage). So the paper designs one input format that works for both.

### 5.1 WordPiece tokens

Text is split with **WordPiece** using a vocabulary of **30,000** tokens. Common words stay whole; rare words are broken into pieces. Pieces that continue a word start with `##`:

> "playing" → `play`, `##ing`
> "embeddings" → `em`, `##bed`, `##ding`, `##s`

This way BERT never meets a truly unknown word — it can always fall back to smaller pieces.

### 5.2 Two special tokens

- **`[CLS]`** ("classification") is always the **first** token. After going through all 12 layers, its output vector (the paper calls it **C**) acts as a summary of the whole input. For classification tasks, this is the vector you put a classifier on.
- **`[SEP]`** ("separator") marks the end of a sentence, and sits between sentence A and sentence B when there are two.

A "sentence" in the paper just means any chunk of text, not necessarily a grammatical sentence.

### 5.3 Three embeddings, added together

Each token's input vector is the **sum** of three embeddings (this is Figure 2 in the paper):

| Input | `[CLS]` | my | dog | is | cute | `[SEP]` | he | likes | play | ##ing | `[SEP]` |
|---|---|---|---|---|---|---|---|---|---|---|---|
| **Token embedding** — *which token?* | E[CLS] | E_my | E_dog | E_is | E_cute | E[SEP] | E_he | E_likes | E_play | E_##ing | E[SEP] |
| **Segment embedding** — *sentence A or B?* | A | A | A | A | A | A | B | B | B | B | B |
| **Position embedding** — *where in the sequence?* | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |

Why the position embedding? Self-attention by itself has no sense of order — "dog bites man" and "man bites dog" would look the same. Position embeddings tell the model where each token sits. BERT learns these, up to a maximum of **512 tokens**.

Why the segment embedding? So the model always knows which tokens belong to the first text and which to the second.

## 6. Pre-training task #1 — Masked Language Model (MLM)

### The basic rule

In every input sequence, **15% of the tokens** are chosen at random. BERT has to predict the original token at each chosen position, using everything around it. Only those 15% are predicted — not the whole sentence.

### The clever twist: 80 / 10 / 10

There's a problem. During pre-training, BERT sees lots of `[MASK]` tokens. During fine-tuning on a real task, it **never** sees `[MASK]` — real sentences don't contain it. That's a mismatch between how it was trained and how it's used.

To soften this, a chosen token isn't always replaced by `[MASK]`. Using the paper's example sentence "my dog is hairy", where "hairy" was chosen:

| How often | What happens | Input becomes |
|---|---|---|
| **80%** | Replace with `[MASK]` | my dog is `[MASK]` |
| **10%** | Replace with a **random** word | my dog is `apple` |
| **10%** | Leave it **unchanged** | my dog is `hairy` |

In all three cases, BERT must output "hairy" at that position.

**Why this works:** BERT never knows which words it will be tested on or which words might have been swapped for random ones. So it can't get lazy — it has to build a good, context-aware understanding of **every** token, not just the `[MASK]` ones. And since random replacement only affects 10% of 15% = **1.5%** of tokens, it doesn't confuse the model's sense of language.

### The cost

A normal language model gets a learning signal from every word. MLM only learns from 15% of words per pass, so it needs more training. The paper checked this (Appendix C.1): MLM does learn slightly slower at first, but it overtakes the left-to-right model almost immediately in final accuracy.

## 7. Pre-training task #2 — Next Sentence Prediction (NSP)

### Why a second task?

Many important tasks are about the **relationship between two texts**: Does this answer match this question? Does sentence B follow logically from sentence A? Filling in blanks within text doesn't directly teach this.

### How it works

For each training example, pick a sentence A, then pick B:

- **50%** of the time, B is the **actual next sentence** in the document → label `IsNext`
- **50%** of the time, B is a **random sentence** from elsewhere → label `NotNext`

BERT uses the `[CLS]` output vector **C** to predict the label. The paper's examples (note MLM masking is applied at the same time):

```
Input = [CLS] the man went to [MASK] store [SEP] he bought a gallon [MASK] milk [SEP]
Label = IsNext

Input = [CLS] the man [MASK] to the store [SEP] penguin [MASK] are flight ##less birds [SEP]
Label = NotNext
```

The final model gets **97–98%** accuracy on this task. The training loss is simply the MLM loss + the NSP loss, so BERT learns both games at the same time.

> **A note from after the paper:** Later research (notably RoBERTa, 2019) found that NSP isn't really necessary and that removing it — while training longer on more data — works as well or better. The BERT paper's own ablation (Section 9 below) showed NSP helped *in their setup*. Both are worth knowing.

### A caveat the paper itself mentions

The `[CLS]` vector C is **not** a meaningful sentence summary until you fine-tune. It was trained for NSP only. So if you want sentence embeddings without fine-tuning, `[CLS]` from the raw pre-trained BERT is a poor choice. (Keep this in mind for the practical part.)

## 8. Pre-training data and cost

| Detail | Value |
|---|---|
| Data | BooksCorpus (800M words) + English Wikipedia (2,500M words, text only — no lists, tables, headers) |
| Why document-level text? | NSP and long sequences need real, consecutive passages — a shuffled bag of single sentences wouldn't work |
| Batch size | 256 sequences × 512 tokens ≈ 128,000 tokens per batch |
| Training steps | 1,000,000 (≈ 40 passes over the data) |
| Optimizer | Adam, learning rate 1e-4, warmup for first 10,000 steps then linear decay, dropout 0.1 |
| Speed trick | First 90% of steps use sequences of length 128, last 10% use 512 (attention cost grows with the *square* of the length) |
| Hardware | BERT-Base: 16 TPU chips. BERT-Large: 64 TPU chips. **4 days each.** |

**The takeaway:** pre-training is very expensive, but it only has to be done **once**. Google released the trained weights, so everyone else just downloads them and does the cheap part — fine-tuning. That's exactly what we'll do in Part B.

## 9. Fine-tuning — adapting BERT to a real task

### The beautiful part: almost nothing changes

To fine-tune, you:

1. Start from the pre-trained BERT weights.
2. Format your input as sentence A (and optionally sentence B).
3. Add **one small output layer** suited to your task.
4. Train **all** the weights (BERT + the new layer) on your labeled data for a few epochs.

The paper shows that sentence A / sentence B maps neatly onto many tasks:

| Task type | Sentence A | Sentence B | What the output layer uses |
|---|---|---|---|
| Sentiment / topic classification | the text | *(empty)* | `[CLS]` vector C |
| Paraphrase detection | sentence 1 | sentence 2 | `[CLS]` vector C |
| Natural language inference (does B follow from A?) | premise | hypothesis | `[CLS]` vector C |
| Question answering | question | passage | every passage token's vector |
| Named entity recognition (tag each word) | the text | *(empty)* | every token's vector |

Note one neat consequence: by putting both texts into one sequence, self-attention automatically lets every word of the question look at every word of the passage and vice versa. Older systems needed special "cross-attention" machinery for that; BERT gets it for free.

### Example 1: classification

The **only** new parameters are a weight matrix $W$ of size $K \times H$, where $K$ is the number of labels. The prediction is

$$P(\text{label}) = \text{softmax}(C\,W^\top)$$

For 2-class sentiment with BERT-Base that's just 2 × 768 = **1,536 new numbers**, on top of 110 million pre-trained ones.

### Example 2: question answering (SQuAD)

Given a question and a passage, find the answer **span** inside the passage. BERT learns just two new vectors:

- a **start vector** $S$ and an **end vector** $E$ (each of size H)

For each passage token $i$ with output vector $T_i$, the score for "the answer starts here" is $S \cdot T_i$, turned into probabilities with a softmax over all tokens. Same for the end with $E$. The predicted answer is the span (start ≤ end) with the highest total score.

For **SQuAD 2.0**, where some questions have *no* answer in the passage, the trick is simple: "no answer" is represented as an answer that starts and ends at the `[CLS]` token.

### Fine-tuning is cheap

The paper says every result in it can be reproduced in **at most 1 hour on a single Cloud TPU**, or a few hours on a GPU. Recommended settings:

| Hyperparameter | Values that work well |
|---|---|
| Batch size | 16 or 32 |
| Learning rate (Adam) | 5e-5, 3e-5, 2e-5 |
| Epochs | 2, 3, or 4 |

Large datasets (100k+ examples) aren't very sensitive to these choices; small datasets are, so try a few combinations.

## 10. Results and what the experiments prove

### 10.1 Headline results — new state of the art on 11 tasks

| Benchmark | What it tests | Previous best | BERT-Large |
|---|---|---|---|
| **GLUE** (official score) | a suite of 9 understanding tasks | 72.8 (OpenAI GPT) | **80.5** |
| **MultiNLI** accuracy | does sentence B follow from A? | 82.1 | **86.7** |
| **SQuAD v1.1** test F1 | find the answer in a passage | 91.7 (top leaderboard ensemble) | **93.2** |
| **SQuAD v2.0** test F1 | same, but some questions have no answer | 78.0 | **83.1** |
| **SWAG** test accuracy | pick the most sensible continuation | 78.0 (GPT) | **86.3** |

On SWAG, BERT-Large even beats one of the human-expert baselines reported for the dataset.

### 10.2 Ablation: what actually caused the improvement? (Table 5)

An **ablation** means removing one ingredient to see how much it mattered. All rows below use BERT-Base size, same data:

| Variant | MNLI | QNLI | MRPC | SST-2 | SQuAD F1 |
|---|---|---|---|---|---|
| BERT-Base (MLM + NSP) | 84.4 | 88.4 | 86.7 | 92.7 | 88.5 |
| No NSP | 83.9 | 84.9 | 86.5 | 92.6 | 87.9 |
| Left-to-right + no NSP (≈ GPT-style) | 82.1 | 84.3 | 77.5 | 92.1 | 77.8 |

How to read this:

- Removing **NSP** hurts the tasks about sentence pairs (QNLI, MNLI, SQuAD).
- Going from **bidirectional to left-to-right** hurts *everything*, and drops SQuAD by over 10 points. Makes sense: for finding an answer span, a token's meaning often depends on what comes after it.

**Conclusion: bidirectionality is the biggest single win.**

### 10.3 Bigger is better — even for small tasks (Table 6)

They trained models from 3 layers up to 24. Accuracy went up steadily with size on every task, **including MRPC, which has only about 3,600 training examples**. It was known that bigger models help on huge tasks; BERT was the first to convincingly show that a *sufficiently pre-trained* big model also helps on tiny ones.

### 10.4 BERT works without fine-tuning too (Table 7)

On named-entity recognition, they also tried the **feature-based** approach: freeze BERT, extract its vectors, and feed them into a separate small model. The best option (concatenating the top 4 layers' vectors) scored 96.1 F1, only **0.3** behind full fine-tuning. So BERT is useful even when you can't or don't want to fine-tune — for example, when you want to compute vectors once and reuse them many times.

### 10.5 BERT vs GPT vs ELMo at a glance

| | ELMo | OpenAI GPT | BERT |
|---|---|---|---|
| Building block | LSTM | Transformer | Transformer |
| Direction | left→right and right→left, trained **separately**, then glued | left→right only | **both directions jointly, in every layer** |
| Pre-training objective | predict next word (each direction) | predict next word | fill in masked words + next sentence prediction |
| How you reuse it | feature-based | fine-tuning | fine-tuning (or feature-based) |
| Good at generating text? | no | **yes** | no |

## Recap — the whole paper on one screen

1. **Goal:** learn general language understanding from unlabeled text, then reuse it everywhere.
2. **Key insight:** existing models only read in one direction; real understanding needs both sides of every word, in every layer.
3. **The obstacle:** a bidirectional model trained to predict words can "see" the answer.
4. **The fix — MLM:** hide 15% of tokens and predict them (with the 80/10/10 trick to reduce the pre-train vs fine-tune mismatch).
5. **Second task — NSP:** predict whether B really follows A, to learn sentence relationships.
6. **Input format:** `[CLS] A [SEP] B [SEP]`, with token + segment + position embeddings summed.
7. **Fine-tuning:** same model + one tiny output layer, trained for 2–4 epochs. Hours, not days.
8. **Results:** new state of the art on 11 tasks; ablations show bidirectionality is the biggest reason.

---

## Check your understanding

Try to answer each one in your own words first, then click to reveal.

<details>
<summary><b>Q1.</b> Why can't you train a deep bidirectional model by simply asking it to predict the next word?</summary>

Because if every word can attend to every other word, the word it is supposed to predict is visible in the input. Through the layers, the model can just copy it. The task becomes trivial and the model learns nothing useful. Masking hides the target so there's nothing to copy.
</details>

<details>
<summary><b>Q2.</b> If BERT replaced chosen tokens with [MASK] 100% of the time, what problem would that cause?</summary>

`[MASK]` never appears in real data at fine-tuning time. The model might learn to produce its best, most context-aware representations only at `[MASK]` positions and treat normal visible words lazily. The 10% random / 10% unchanged cases mean BERT can never be sure a visible word is "safe", so it builds good representations for every token. The paper's Table 8 shows 100%-[MASK] works noticeably worse in the feature-based setting.
</details>

<details>
<summary><b>Q3.</b> You want to use BERT to decide whether two customer questions mean the same thing. How would you format the input, and which output vector would the classifier use?</summary>

`[CLS] question 1 [SEP] question 2 [SEP]`, with question 1 getting segment A embeddings and question 2 getting segment B. The classifier sits on top of the `[CLS]` output vector C. (This is exactly the QQP task in GLUE.)
</details>

<details>
<summary><b>Q4.</b> In the ablation, which change hurt SQuAD question answering the most, and why does that make intuitive sense?</summary>

Switching from bidirectional (MLM) to left-to-right training: SQuAD F1 dropped from 87.9 to 77.8. To decide whether a token is the start or end of the answer, you often need to see the words that come after it — and a left-to-right model's token vectors have no right-side context at all.
</details>

<details>
<summary><b>Q5.</b> Why was BERT-Base made exactly the same size as OpenAI GPT?</summary>

To make a fair comparison. If BERT were bigger, you couldn't tell whether it won because of size or because of its ideas. With matched size, the gains can be attributed to bidirectionality and the pre-training tasks.
</details>

<details>
<summary><b>Q6.</b> Predict: if you take raw pre-trained BERT (no fine-tuning) and use the [CLS] vector as a "meaning of the sentence" embedding to find similar sentences, will it work well?</summary>

Probably not very well. The paper explicitly warns that C isn't a meaningful sentence representation without fine-tuning, because it was only trained for next sentence prediction. For sentence similarity, people either fine-tune or use models built specifically for sentence embeddings (e.g. Sentence-BERT).
</details>

<details>
<summary><b>Q7.</b> Can BERT write a story for you the way a chatbot does? Why or why not?</summary>

Not naturally. BERT is an encoder: it was trained to fill in blanks using both sides, not to produce text one word at a time from left to right. Generating text needs the left-only (decoder) setup that GPT uses.
</details>

# PART B — Using BERT in practice

Now we'll **see** each idea from the paper working, using Google's real pre-trained BERT weights through the [Hugging Face `transformers`](https://huggingface.co/docs/transformers) library.

| Section | Paper idea you'll see |
|---|---|
| 12 | WordPiece, `[CLS]`, `[SEP]`, segment IDs (Section 5) |
| 13 | Masked Language Model — the fill-in-the-blank game (Section 6) |
| 14 | Next Sentence Prediction (Section 7) |
| 15 | Contextual, bidirectional embeddings (Sections 1–2) |
| 16 | Fine-tuned BERT models for real tasks (Section 9) |
| 17 | Fine-tuning BERT yourself (Section 9) |

**Requirements:** Python 3.9+ and an internet connection (the first run downloads the model, roughly 440 MB). A GPU is *not* needed — everything here runs on a laptop CPU. Google Colab works great too.

## 11. Setup

In [ ]:
# Run once. In Colab these are already installed, but this is harmless.
!pip install -q transformers torch

In [ ]:
import torch
from transformers import AutoTokenizer, pipeline

MODEL_NAME = "bert-base-uncased"   # the original BERT-Base from the paper (lowercased English)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)   # the paper says ~30,000 WordPiece tokens

## 12. See BERT's input format (paper Section 5)

Let's feed the exact sentence pair from the paper's Figure 2 into the tokenizer and look at what BERT actually receives.

In [ ]:
encoded = tokenizer("my dog is cute", "he likes playing")

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print(f"{'position':<10}{'token':<10}{'segment':<8}")
for pos, (tok, seg) in enumerate(zip(tokens, encoded["token_type_ids"])):
    print(f"{pos:<10}{tok:<10}{'A' if seg == 0 else 'B':<8}")

Compare this to the table in Section 5.3:

- `[CLS]` was added at the start and `[SEP]` after each sentence — automatically.
- "playing" became `play` + `##ing` (WordPiece).
- `token_type_ids` are the **segment embeddings**: 0 = sentence A, 1 = sentence B.
- The position number is the **position embedding** index.

Try a rare or made-up word to see WordPiece break it into pieces:

In [ ]:
print(tokenizer.tokenize("Transformers are unbelievably good at tokenization"))

## 13. The fill-in-the-blank game — Masked LM (paper Section 6)

The raw pre-trained BERT still has its MLM "head" (the layer that predicts the hidden word). The `fill-mask` pipeline lets us play the same game BERT was trained on.

In [ ]:
fill_mask = pipeline("fill-mask", model=MODEL_NAME)

def show_guesses(sentence, k=5):
    print(sentence)
    for guess in fill_mask(sentence, top_k=k):
        print(f"   {guess['token_str']:<12} {guess['score']:.3f}")
    print()

show_guesses("The capital of France is [MASK].")
show_guesses("My dog is [MASK].")   # the paper's own example sentence

### Why bidirectional matters — see it directly

In both sentences below, the words **before** the blank are identical. Only the words **after** it differ. A left-to-right model would have to make the same guess for both. Watch BERT change its answer based on the right-hand context:

In [ ]:
show_guesses("I went to the [MASK] to deposit my paycheck.")
show_guesses("I went to the [MASK] to see the doctor.")
show_guesses("I went to the [MASK] to buy some bread.")

**Try it yourself:** write your own sentences with one `[MASK]`. Some things to test:
- Put the important clue only *after* the blank.
- Ask a factual question ("Albert Einstein was born in [MASK].") — BERT absorbed a lot of facts from Wikipedia.
- Notice that BERT fills in exactly **one token** per `[MASK]`; it's not a text generator (quiz Q7).

In [ ]:
show_guesses("Albert Einstein was born in [MASK].")

## 14. Next Sentence Prediction (paper Section 7)

The original weights also include the NSP head. It reads `[CLS] A [SEP] B [SEP]` and uses the `[CLS]` vector to output two scores: **IsNext** vs **NotNext**.

In [ ]:
from transformers import BertForNextSentencePrediction

nsp_model = BertForNextSentencePrediction.from_pretrained(MODEL_NAME)
nsp_model.eval()

def is_next_probability(sentence_a, sentence_b):
    inputs = tokenizer(sentence_a, sentence_b, return_tensors="pt")
    with torch.no_grad():
        logits = nsp_model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    return probs[0].item()   # index 0 = "IsNext", index 1 = "NotNext"

a = "The man went to the store."
print(f"{is_next_probability(a, 'He bought a gallon of milk.'):.3f}  <- real continuation")
print(f"{is_next_probability(a, 'Penguins are flightless birds.'):.3f}  <- random sentence")

These are the two examples from the paper's appendix. BERT should give a high "IsNext" probability to the first and a low one to the second.

## 15. Contextual embeddings: one word, different meanings (paper Sections 1–2)

Here we use BERT the **feature-based** way (Section 10.4): no fine-tuning, just read the output vectors.

We'll get BERT's 768-number vector for the word "bank" in different sentences, then measure how similar those vectors are with **cosine similarity** (1.0 = pointing in the same direction, near 0 = unrelated).

In [ ]:
from transformers import AutoModel
import torch.nn.functional as F

bert = AutoModel.from_pretrained(MODEL_NAME)   # BERT body only, no task head
bert.eval()

def word_vector(sentence, word):
    # Return BERT's final-layer vector for `word` inside `sentence`.
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = bert(**inputs).last_hidden_state[0]      # shape: (num_tokens, 768)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    return hidden[tokens.index(word)]

money_1 = word_vector("I deposited cash at the bank.", "bank")
money_2 = word_vector("The bank approved my loan application.", "bank")
river   = word_vector("We had a picnic on the bank of the river.", "bank")

sim = lambda x, y: F.cosine_similarity(x, y, dim=0).item()
print(f"money bank  vs money bank : {sim(money_1, money_2):.3f}")
print(f"money bank  vs river bank : {sim(money_1, river):.3f}")

The two "money" banks should be more similar to each other than either is to the "river" bank — even though it's literally the same word. An old word2vec-style model would give a similarity of exactly 1.0 for all three, because it has just one vector per word.

**Try it yourself:** replace "bank" with another ambiguous word — e.g. "bat" (animal vs baseball), "spring" (season vs coil), "light" (not heavy vs brightness).

## 16. Using BERT models that were already fine-tuned (paper Section 9)

In real life you rarely fine-tune from scratch for common tasks — people have shared thousands of fine-tuned BERT models on the Hugging Face Hub. Each one is exactly what Section 9 describes: pre-trained BERT + one small output layer, trained on a task dataset.

### 16a. Question answering — the SQuAD setup

This model is BERT fine-tuned on SQuAD 2.0. Under the hood it's doing exactly what the paper describes: scoring every passage token with a **start vector** and an **end vector**, then returning the best span.

In [ ]:
qa = pipeline("question-answering", model="deepset/bert-base-cased-squad2")

passage = (
    "BERT was introduced by researchers at Google AI Language in 2018. "
    "It was pre-trained on BooksCorpus and English Wikipedia, which together contain "
    "about 3.3 billion words. Pre-training BERT-Large took four days on 64 TPU chips."
)

for question in ["Who introduced BERT?",
                 "How many words were in the training data?",
                 "How long did pre-training take?"]:
    result = qa(question=question, context=passage)
    print(f"Q: {question}\nA: {result['answer']}   (score {result['score']:.2f}, "
          f"characters {result['start']}-{result['end']})\n")

Notice the answer is always a **span copied from the passage** (with start and end positions) — BERT doesn't write new text, it points at where the answer is.

### 16b. Sentiment classification — the `[CLS]` + classifier setup

This model is multilingual BERT fine-tuned on product reviews to predict 1–5 stars. Here the output layer sits on the `[CLS]` vector, just like the GLUE setup.

In [ ]:
sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

reviews = [
    "Absolutely loved it, would buy again!",
    "It's okay, nothing special.",
    "Broke after two days. Total waste of money.",
    "Ce produit est excellent.",              # French - it's a multilingual BERT
]
for review, pred in zip(reviews, sentiment(reviews)):
    print(f"{pred['label']:<8} ({pred['score']:.2f})  {review}")

## 17. Fine-tune BERT yourself (a tiny, toy version)

This is the full recipe from Section 9 in about 20 lines: load pre-trained BERT, add a classification layer, train everything briefly.

When you run the first cell you'll see a warning like *"Some weights of BertForSequenceClassification were not initialized… classifier.weight, classifier.bias"*. **That's not an error — that is literally the paper's new $W$ matrix** being created with random values. Everything else comes from pre-training.

> We use only 16 example sentences so it finishes in under a minute on a CPU. Real fine-tuning uses thousands of examples, but the steps are identical. With such tiny data we train for more epochs than the paper's 2–4.

In [ ]:
from transformers import AutoModelForSequenceClassification

torch.manual_seed(0)

# Toy task: is a sentence about FOOD (1) or SPORTS (0)?
train_texts = [
    "The pasta was cooked perfectly with fresh basil.",  "She scored two goals in the final match.",
    "This soup needs more salt and pepper.",             "The team won the championship last night.",
    "I baked a chocolate cake for dessert.",             "He ran the marathon in under three hours.",
    "The sushi at that restaurant is amazing.",          "The coach called a timeout in the last quarter.",
    "Grilled vegetables taste great with olive oil.",    "Our goalkeeper saved a penalty kick.",
    "We ordered spicy noodles and dumplings.",           "The tennis player served an ace to win the set.",
    "Fresh bread smells wonderful in the morning.",      "The referee showed him a red card.",
    "This curry is rich and full of flavor.",            "She broke the world record in swimming.",
]
train_labels = [1, 0] * 8

classifier = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

In [ ]:
optimizer = torch.optim.AdamW(classifier.parameters(), lr=5e-5)   # a learning rate from the paper's range
batch = tokenizer(train_texts, padding=True, return_tensors="pt")
labels = torch.tensor(train_labels)

classifier.train()
for epoch in range(10):
    outputs = classifier(**batch, labels=labels)   # forward pass; the library computes the loss
    outputs.loss.backward()                        # compute gradients for ALL weights (BERT + W)
    optimizer.step()                               # update the weights
    optimizer.zero_grad()
    print(f"epoch {epoch + 1:>2}   loss {outputs.loss.item():.4f}")

The loss should go down — the model is learning the task. Now test it on sentences it has **never seen**. Notice none of them reuse the exact training vocabulary; BERT generalizes because of what it learned during pre-training.

In [ ]:
test_texts = [
    "I had a delicious burrito for lunch.",
    "The striker missed an easy shot.",
    "Mix the flour, sugar, and eggs in a bowl.",
    "The cyclists raced up the mountain.",
]

classifier.eval()
with torch.no_grad():
    probs = torch.softmax(classifier(**tokenizer(test_texts, padding=True, return_tensors="pt")).logits, dim=1)

for text, p in zip(test_texts, probs):
    label = "FOOD" if p[1] > 0.5 else "SPORTS"
    print(f"{label:<7} ({max(p).item():.2f})  {text}")

## Where to go next

**Ideas to try on your own**
- In Section 17, swap the toy data for a real dataset (e.g. IMDB reviews via the `datasets` library) and use the Hugging Face `Trainer`.
- In Section 15, compare vectors from different layers (`bert(..., output_hidden_states=True)`) — the paper found the top 4 layers combined work best as features.
- Try `bert-large-uncased` and see whether the fill-mask guesses improve (Section 10.3: bigger is better).

**Models that built on BERT (worth knowing the names)**
- **RoBERTa** (2019): same architecture, drops NSP, trains longer on more data → better.
- **DistilBERT** (2019): a smaller, faster BERT that keeps most of the accuracy.
- **ALBERT** (2019): shares weights between layers to cut parameters.
- **Sentence-BERT** (2019): fine-tunes BERT specifically to produce good sentence embeddings (fixes the problem from quiz Q6).

**Further reading**
- The paper: https://arxiv.org/abs/1810.04805
- Original code and weights: https://github.com/google-research/bert
- *The Illustrated BERT* by Jay Alammar — an excellent visual explanation.
- *The Annotated Transformer* (Harvard NLP) — if you later *do* want to see the architecture in code.